
# Moscow Macro Parser — quarterly macro dataset for the market block

Этот ноутбук собирает **квартальные макроэкономические ряды для Москвы** в один `CSV`, который потом можно использовать как вход для `market block`.

## Что попадает в итоговый CSV

- **Ключевая ставка ЦБ**: средняя за квартал, значение на конец квартала, изменение к/к.
- **Курсы ЦБ**: USD/RUB и EUR/RUB — средние за квартал, конец квартала, лог-доходности.
- **Ипотека по Москве (ЦБ, региональный разрез)**:
  - общее число ипотечных кредитов,
  - общий объём ипотечных кредитов,
  - средневзвешенная ставка,
  - аналогичные ряды по ипотеке под **ДДУ**.
- **Индекс московской недвижимости Домклик / MOEX (`MREDC`)**.
- **Дополнительные биржевые индексы MOEX** (по желанию): `IMOEX`, `RGBI`.
- **Официальные индексы и статистика Мосстата**:
  - индекс цен на первичном рынке жилья,
  - ИПЦ по Москве,
  - ввод жилья в Москве.
- **Опциональные policy-флаги** по льготной ипотеке.

## Идея

В market block лучше не передавать всю эту сырую макроинформацию напрямую в сделочную hedonic-модель.  
Обычно удобнее сначала собрать и смоделировать отдельный квартальный `market_state`, а уже потом отдавать в hedonic один-два агрегированных признака (`market_log_price`, `market_return`).

Но сначала нужен **чистый и воспроизводимый квартальный макро-датасет** — именно его и строит этот ноутбук.



## Источники

Официальные источники, под которые написаны парсеры:

- Банк России — ключевая ставка: `https://www.cbr.ru/hd_base/keyrate/`
- Банк России — официальные курсы валют: `https://www.cbr.ru/scripts/XML_daily_dyn.asp`
- Банк России — ипотечная статистика по регионам и по ДДУ: `https://www.cbr.ru/statistics/bank_sector/mortgage/`
- MOEX ISS — история индексов: `https://iss.moex.com/iss/history/engines/stock/markets/index/securities/{SECID}.json`
- MOEX — карточка индекса `MREDC`: `https://www.moex.com/ru/index/MREDC`
- Мосстат — цены и тарифы / индексы цен на первичном рынке жилья
- Мосстат — ИПЦ по Москве
- Мосстат — ввод жилья в Москве

Парсеры сделаны так, чтобы при возможности брать данные **напрямую из файла/endpoint**, а для Мосстата — через страницу-раздел с поиском актуального `xlsx` по маске в названии.


In [1]:

from __future__ import annotations

import io
import json
import math
import re
import warnings
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from typing import Iterable, Optional
from urllib.parse import urljoin, urlencode

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 200
pd.options.display.width = 240


In [2]:
# ── Конфигурация ─────────────────────────────────────────────────────────────
PROJECT_DIR = Path("./cashflow_project")
OUT_DIR = PROJECT_DIR / "data" / "macro"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Берём максимально длинную историю для макро-блока.
# Ключевая ставка доступна с 17.09.2013, валютные курсы и часть ипотечных рядов — раньше.
START_DATE = "2013-01-01"
END_DATE   = None  # None => today
REGION_NAME = "г. Москва"
REGION_SHORT = "Moscow"

OUTPUT_CSV = OUT_DIR / f"macro_quarterly_{REGION_SHORT}.csv"
OUTPUT_COVERAGE_CSV = OUT_DIR / f"macro_quarterly_{REGION_SHORT}_coverage.csv"

# Для XML-курсов ЦБ
CBR_VAL_IDS = {
    "USD": "R01235",
    "EUR": "R01239",
}

# Полная история ключевой ставки через query-параметры страницы ЦБ.
# Именно так мы избегаем дефолтной "короткой витрины" за последние дни.
KEY_RATE_HISTORY_URL = "https://www.cbr.ru/hd_base/KeyRate/"

# Ипотека ЦБ: региональные ряды
CBR_MORTGAGE_URLS = {
    "mortgage_total_count": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_10_Quantity_mortgage.xlsx",
        "sheet": "в рублях",
        "value_name": "mortgage_total_count_monthly",
    },
    "mortgage_total_volume": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_11_New_loans_mortgage.xlsx",
        "sheet": "в рублях",
        "value_name": "mortgage_total_volume_mln_rub_monthly",
    },
    "mortgage_total_rate": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_13_Rates_mortgage.xlsx",
        "sheet": "ставка в рублях",
        "value_name": "mortgage_total_rate_pct_monthly",
    },
    "mortgage_ddu_count": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_15_Quantity_scpa_mortgage.xlsx",
        "sheet": "в рублях",
        "value_name": "mortgage_ddu_count_monthly",
    },
    "mortgage_ddu_volume": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_16_New_loans_scpa_mortgage.xlsx",
        "sheet": "в рублях",
        "value_name": "mortgage_ddu_volume_mln_rub_monthly",
    },
    "mortgage_ddu_rate": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_17_Rates_scpa_mortgage.xlsx",
        "sheet": "ставка в рублях",
        "value_name": "mortgage_ddu_rate_pct_monthly",
    },
}

# MOEX ISS
MOEX_ISS_HISTORY_URL = "https://iss.moex.com/iss/history/engines/stock/markets/index/securities/{secid}.json"
MOEX_INDEX_SECIDS = ["MREDC", "IMOEX", "RGBI"]

# Мосстат: страницы-разделы, с которых ищем актуальные XLSX
MOSSTAT_PAGES = {
    "primary_price_index": "https://77.rosstat.gov.ru/folder/64508",
    "cpi": "https://77.rosstat.gov.ru/folder/64640",
    "housing_completion": "https://77.rosstat.gov.ru/folder/64519",
}

# Поисковые маски по href / тексту ссылки
MOSSTAT_PATTERNS = {
    "primary_price_index": [r"первичн.*рынк.*жиль", r"индекс.*цен"],
    "cpi": [r"индекс.*потребительск", r"товар.*услуг"],
    "housing_completion": [r"ввод.*жил.*дом"],
}

REQUEST_TIMEOUT = 60
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "cashflow-macro-parser/1.1 (research notebook)"
})

print("OUTPUT_CSV:", OUTPUT_CSV)

OUTPUT_CSV: cashflow_project\data\macro\macro_quarterly_Moscow.csv


## Вспомогательные функции

In [3]:

RU_MONTHS = {
    "январь": 1, "февраль": 2, "март": 3, "апрель": 4,
    "май": 5, "июнь": 6, "июль": 7, "август": 8,
    "сентябрь": 9, "октябрь": 10, "ноябрь": 11, "декабрь": 12,
}


def parse_ru_month_year(text: object) -> pd.Timestamp:
    if pd.isna(text):
        return pd.NaT
    s = str(text).strip().lower().replace("ё", "е")
    m = re.match(r"([а-я]+)\s+(\d{4})", s)
    if not m:
        return pd.NaT
    month = RU_MONTHS.get(m.group(1))
    year = int(m.group(2))
    if month is None:
        return pd.NaT
    return pd.Timestamp(year=year, month=month, day=1)


def to_numeric_safe(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace("\xa0", " ", regex=False).str.replace(" ", " ", regex=False).str.strip()
    s = s.str.replace("%", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    out = pd.to_numeric(s, errors="coerce")

    # На отдельных витринах ЦБ ставка иногда приезжает как 1500 вместо 15.00.
    # Аккуратно нормализуем только явно "процентные" масштабы.
    out = out.where(~(out > 100), out / 100.0)
    return out


def clip_date_range(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    start_ts = pd.Timestamp(START_DATE)
    end_ts = pd.Timestamp.today().normalize() if END_DATE is None else pd.Timestamp(END_DATE)
    return out[(out[date_col] >= start_ts) & (out[date_col] <= end_ts)].copy()


def add_quarter_columns(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    out["quarter"] = out[date_col].dt.to_period("Q").astype(str)
    out["quarter_start"] = out[date_col].dt.to_period("Q").dt.start_time
    out["quarter_end"] = out[date_col].dt.to_period("Q").dt.end_time
    return out


def aggregate_daily_to_quarter(df: pd.DataFrame, value_col: str, prefix: str) -> pd.DataFrame:
    tmp = add_quarter_columns(df, "date")
    q = (
        tmp.groupby("quarter", as_index=False)
        .agg(
            **{
                f"{prefix}_avg_q": (value_col, "mean"),
                f"{prefix}_eoq": (value_col, lambda s: s.dropna().iloc[-1] if s.dropna().shape[0] else np.nan),
                f"{prefix}_min_q": (value_col, "min"),
                f"{prefix}_max_q": (value_col, "max"),
                f"{prefix}_obs_n": (value_col, "size"),
            }
        )
        .sort_values("quarter")
        .reset_index(drop=True)
    )
    if f"{prefix}_eoq" in q.columns:
        q[f"{prefix}_log_return_q"] = np.log(q[f"{prefix}_eoq"]) - np.log(q[f"{prefix}_eoq"].shift(1))
        q[f"{prefix}_qoq_change"] = q[f"{prefix}_eoq"] - q[f"{prefix}_eoq"].shift(1)
    return q


def aggregate_monthly_to_quarter(df: pd.DataFrame, value_col: str, prefix: str, how: str = "sum") -> pd.DataFrame:
    tmp = add_quarter_columns(df, "date")
    if how == "sum":
        val = (value_col, "sum")
    elif how == "mean":
        val = (value_col, "mean")
    elif how == "last":
        val = (value_col, lambda s: s.dropna().iloc[-1] if s.dropna().shape[0] else np.nan)
    else:
        raise ValueError(f"Unsupported how={how}")

    q = (
        tmp.groupby("quarter", as_index=False)
        .agg(**{f"{prefix}": val})
        .sort_values("quarter")
        .reset_index(drop=True)
    )
    q[f"{prefix}_qoq_change"] = q[f"{prefix}"].diff()
    if (q[f"{prefix}"] > 0).all():
        q[f"{prefix}_log_return_q"] = np.log(q[f"{prefix}"]) - np.log(q[f"{prefix}"].shift(1))
    return q


def make_quarter_spine(start: str = START_DATE, end: Optional[str] = END_DATE) -> pd.DataFrame:
    start_ts = pd.Timestamp(start).to_period("Q").start_time
    end_ts = (pd.Timestamp.today() if end is None else pd.Timestamp(end)).to_period("Q").start_time
    qs = pd.period_range(start=start_ts, end=end_ts, freq="Q")
    return pd.DataFrame({"quarter": qs.astype(str)})


def outer_merge_on_quarter(frames: list[pd.DataFrame]) -> pd.DataFrame:
    out = make_quarter_spine()
    for df in frames:
        if df is None or df.empty:
            continue
        cols = [c for c in df.columns if c == "quarter" or c not in out.columns]
        out = out.merge(df[cols], on="quarter", how="left")
    return out


def build_coverage_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        if col == "quarter":
            continue
        s = df[["quarter", col]].dropna()
        rows.append({
            "feature": col,
            "first_quarter": s["quarter"].iloc[0] if len(s) else None,
            "last_quarter": s["quarter"].iloc[-1] if len(s) else None,
            "non_null_quarters": int(len(s)),
        })
    return pd.DataFrame(rows).sort_values(["first_quarter", "feature"], na_position="last").reset_index(drop=True)


## 1. Ключевая ставка ЦБ

In [4]:
def fetch_key_rate_history(start: str = START_DATE, end: str | None = END_DATE) -> pd.DataFrame:
    """
    Полная история ключевой ставки ЦБ.

    Почему предыдущая версия падала:
    - страница ЦБ часто отдаёт не обычную HTML-таблицу, а разметку, где pandas.read_html
      не всегда стабильно распознаёт заголовки;
    - иногда query-параметры диапазона игнорируются;
    - в результате таблицы находились, но нужные колонки не распознавались,
      поэтому last_err оставался None.

    Эта версия:
    1) явно запрашивает страницу с диапазоном дат;
    2) сначала пытается распарсить таблицу через read_html;
    3) если не получилось — парсит строки вида DD.MM.YYYY RATE регуляркой прямо из HTML-текста;
    4) если и это не удалось — делает fallback на список дат изменения ставки из XLSX,
       а затем разворачивает его в daily step function.
    """
    import re

    start_ts = pd.Timestamp(start).normalize()
    end_ts = pd.Timestamp.today().normalize() if end is None else pd.Timestamp(end).normalize()

    params = {
        'UniDbQuery.Posted': 'True',
        'UniDbQuery.From': start_ts.strftime('%d.%m.%Y'),
        'UniDbQuery.To': end_ts.strftime('%d.%m.%Y'),
    }
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Accept-Language': 'ru,en;q=0.9',
    }

    def _finalize_changes(df_changes: pd.DataFrame) -> pd.DataFrame:
        out = df_changes.copy()
        out['date'] = pd.to_datetime(out['date'], dayfirst=True, errors='coerce')
        out['key_rate_pct'] = to_numeric_safe(out['key_rate_pct'])
        out = out.dropna(subset=['date', 'key_rate_pct']).sort_values('date').drop_duplicates('date')
        out = out[out['date'] >= start_ts]
        if out.empty:
            raise RuntimeError('История ключевой ставки получена пустой после очистки.')

        daily_index = pd.date_range(out['date'].min(), end_ts, freq='D')
        stepped = pd.DataFrame({'date': daily_index})
        stepped = stepped.merge(out, on='date', how='left')
        stepped['key_rate_pct'] = stepped['key_rate_pct'].ffill()
        stepped = stepped[stepped['date'] >= start_ts].reset_index(drop=True)
        return stepped

    # --- 1) Основной путь: HTML-страница ключевой ставки ---
    for url in [KEY_RATE_HISTORY_URL, 'https://www.cbr.ru/eng/hd_base/KeyRate/']:
        try:
            resp = SESSION.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            html = resp.text

            # 1a. пробуем read_html
            try:
                tables = pd.read_html(io.StringIO(html))
            except Exception:
                tables = []

            for df in tables:
                cur = df.copy()
                if isinstance(cur.columns, pd.MultiIndex):
                    cur.columns = [' '.join([str(x) for x in tup if str(x) != 'nan']).strip() for tup in cur.columns]
                cur.columns = [str(c).strip().lower() for c in cur.columns]

                date_col = next((c for c in cur.columns if 'дат' in c or c == 'date'), None)
                rate_col = next((c for c in cur.columns if 'став' in c or c == 'rate'), None)
                if date_col and rate_col:
                    parsed = cur[[date_col, rate_col]].rename(columns={date_col: 'date', rate_col: 'key_rate_pct'})
                    parsed = parsed.dropna(how='all')
                    if len(parsed) >= 3:
                        return _finalize_changes(parsed)

            # 1b. fallback: regex по тексту HTML
            # Ищем строки вида 03.04.2026 15,00 / 03.04.2026 15.00
            pairs = re.findall(r'(\d{2}\.\d{2}\.\d{4})\s+([0-9]+(?:[\.,][0-9]+)?)', html)
            if pairs:
                parsed = pd.DataFrame(pairs, columns=['date', 'key_rate_pct'])
                parsed = parsed.drop_duplicates()
                if parsed['date'].nunique() >= 10:
                    return _finalize_changes(parsed)
        except Exception:
            pass

    # --- 2) Fallback: XLSX с датами изменения ставки ---
    # Официальный файл ЦБ с датами изменения ставки денежно-кредитной политики.
    xlsx_urls = [
        'https://www.cbr.ru/vfs/hd_base/procstav/ir_chg_mpo/ir_chg_mpo.xlsx',
        'https://www.cbr.ru/vfs/hd_base/procstav/ir_chg_mpo/ir_chg_mpo_e.xlsx',
    ]
    for xurl in xlsx_urls:
        try:
            chg = pd.read_excel(xurl)
            chg.columns = [str(c).strip().lower() for c in chg.columns]
            date_col = next((c for c in chg.columns if 'date' in c or 'дат' in c), None)
            rate_col = next((c for c in chg.columns if 'key rate' in c or 'ключевая ставка' in c or c == 'key rate'), None)
            if date_col and rate_col:
                parsed = chg[[date_col, rate_col]].rename(columns={date_col: 'date', rate_col: 'key_rate_pct'})
                parsed = parsed.dropna(how='all')
                if len(parsed) >= 3:
                    return _finalize_changes(parsed)
        except Exception:
            pass

    raise RuntimeError(
        'Не удалось получить историю ключевой ставки ни со страницы ЦБ, ни из fallback XLSX. '
        'Проверь доступ к cbr.ru из среды запуска или обнови парсер под текущую разметку страницы.'
    )


key_rate_daily = fetch_key_rate_history()
key_rate_q = aggregate_daily_to_quarter(key_rate_daily, 'key_rate_pct', 'key_rate')
display(key_rate_q.head())
display(key_rate_q.tail())



,quarter,key_rate_avg_q,key_rate_eoq,key_rate_min_q,key_rate_max_q,key_rate_obs_n,key_rate_log_return_q,key_rate_qoq_change
0,2013Q3,5.500000,5.5,5.5,5.5,14,NaN,NaN
1,2013Q4,5.500000,5.5,5.5,5.5,92,0.000000,0.0
2,2014Q1,5.983333,7.0,5.5,7.0,90,0.241162,1.5
3,2014Q2,7.351648,7.5,7.0,7.5,91,0.068993,0.5
4,2014Q3,7.853261,8.0,7.5,8.0,92,0.064539,0.5


,quarter,key_rate_avg_q,key_rate_eoq,key_rate_min_q,key_rate_max_q,key_rate_obs_n,key_rate_log_return_q,key_rate_qoq_change
47,2025Q2,20.758242,20.0,20.0,21.0,91,-0.048790,-1.0
48,2025Q3,18.413043,17.0,17.0,20.0,92,-0.162519,-3.0
49,2025Q4,16.586957,16.0,16.0,17.0,92,-0.060625,-1.0
50,2026Q1,15.705556,15.0,15.0,16.0,90,-0.064539,-1.0
51,2026Q2,15.000000,15.0,15.0,15.0,5,0.000000,0.0


## 2. Официальные курсы ЦБ: USD/RUB и EUR/RUB

In [5]:

def fetch_cbr_fx_daily(char_code: str, val_id: str, start: str = START_DATE, end: Optional[str] = END_DATE) -> pd.DataFrame:
    end_eff = pd.Timestamp.today().strftime("%d/%m/%Y") if end is None else pd.Timestamp(end).strftime("%d/%m/%Y")
    start_eff = pd.Timestamp(start).strftime("%d/%m/%Y")
    url = (
        "https://www.cbr.ru/scripts/XML_dynamic.asp?" +
        urlencode({"date_req1": start_eff, "date_req2": end_eff, "VAL_NM_RQ": val_id})
    )
    r = SESSION.get(url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()

    root = ET.fromstring(r.content)
    rows = []
    for rec in root.findall("Record"):
        d = rec.attrib.get("Date")
        value = rec.findtext("Value")
        nominal = rec.findtext("Nominal")
        rows.append({
            "date": pd.to_datetime(d, dayfirst=True, errors="coerce"),
            f"{char_code.lower()}_rub": pd.to_numeric(str(value).replace(",", "."), errors="coerce"),
            f"{char_code.lower()}_nominal": pd.to_numeric(str(nominal).replace(",", "."), errors="coerce"),
        })

    df = pd.DataFrame(rows).dropna(subset=["date"]).sort_values("date")
    val_col = f"{char_code.lower()}_rub"
    nom_col = f"{char_code.lower()}_nominal"
    df[val_col] = df[val_col] / df[nom_col].replace(0, np.nan)
    df = df[["date", val_col]].copy()
    return clip_date_range(df)


usd_daily = fetch_cbr_fx_daily("USD", CBR_VAL_IDS["USD"])
eur_daily = fetch_cbr_fx_daily("EUR", CBR_VAL_IDS["EUR"])

usd_q = aggregate_daily_to_quarter(usd_daily, "usd_rub", "usd_rub")
eur_q = aggregate_daily_to_quarter(eur_daily, "eur_rub", "eur_rub")

display(usd_q.tail())


,quarter,usd_rub_avg_q,usd_rub_eoq,usd_rub_min_q,usd_rub_max_q,usd_rub_obs_n,usd_rub_log_return_q,usd_rub_qoq_change
49,2025Q2,80.936890,78.4685,78.1959,86.1891,59,-0.064318,-5.2128
50,2025Q3,80.560053,82.8676,77.8855,85.6647,66,0.054547,4.3991
51,2025Q4,79.890828,78.2267,76.0937,83.0000,65,-0.057633,-4.6409
52,2026Q1,78.331987,81.2955,75.7327,84.8379,54,0.038480,3.0688
53,2026Q2,80.484075,79.7293,79.7293,81.2504,4,-0.019454,-1.5662


## 3. Ипотека по Москве из региональной статистики ЦБ

## 3a. Важное замечание по покрытию рядов

У разных источников разная глубина истории.  
В частности:

- ключевая ставка Банка России доступна с **17.09.2013**;
- `MREDC` доступен с **28.12.2016**;
- региональные ипотечные ряды ЦБ по Москве обычно тянутся заметно раньше 2019;
- отдельные ряды по **ипотеке под ДДУ** в актуальных таблицах ЦБ начинаются только с более позднего периода, и это нормально — ранние кварталы там должны оставаться `NaN`, а не заполняться искусственно.

In [6]:

def _find_header_row_with_months(raw: pd.DataFrame) -> int:
    for i in range(min(10, len(raw))):
        parsed = pd.Series(raw.iloc[i, 1:]).apply(parse_ru_month_year)
        if parsed.notna().sum() >= 6:
            return i
    raise RuntimeError("Не удалось найти строку с month-year заголовками")


def parse_cbr_regional_monthly_xlsx(source: str, sheet_name: str, region_name: str, value_name: str) -> pd.DataFrame:
    raw = pd.read_excel(source, sheet_name=sheet_name, header=None)
    header_row = _find_header_row_with_months(raw)

    months = pd.Series(raw.iloc[header_row, 1:]).apply(parse_ru_month_year)
    data = raw.iloc[header_row + 1 :].copy()
    data.columns = ["region"] + list(months)
    data = data.dropna(subset=["region"]).copy()
    data["region"] = data["region"].astype(str).str.strip()

    row = data.loc[data["region"].str.lower() == region_name.lower()].copy()
    if row.empty:
        candidates = data.loc[data["region"].str.contains("моск", case=False, na=False), ["region"]]
        raise RuntimeError(f"Регион {region_name!r} не найден. Кандидаты: {candidates['region'].tolist()[:10]}")

    row = row.drop(columns=["region"]).T.reset_index()
    row.columns = ["date", value_name]
    row["date"] = pd.to_datetime(row["date"], errors="coerce")
    row[value_name] = to_numeric_safe(row[value_name])
    row = row.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return clip_date_range(row)


def fetch_all_mortgage_monthly(region_name: str = REGION_NAME) -> dict[str, pd.DataFrame]:
    out = {}
    for key, meta in CBR_MORTGAGE_URLS.items():
        out[key] = parse_cbr_regional_monthly_xlsx(
            source=meta["url"],
            sheet_name=meta["sheet"],
            region_name=region_name,
            value_name=meta["value_name"],
        )
    return out


mortgage_monthly = fetch_all_mortgage_monthly()
for k, df in mortgage_monthly.items():
    print(k, df.shape, df["date"].min().date(), df["date"].max().date())
    display(df.tail(2))


mortgage_total_count (86, 2) 2019-01-01 2026-02-01


,date,mortgage_total_count_monthly
84,2026-01-01,68.27
85,2026-02-01,47.21


mortgage_total_volume (86, 2) 2019-01-01 2026-02-01


,date,mortgage_total_volume_mln_rub_monthly
84,2026-01-01,562.51
85,2026-02-01,344.74


mortgage_total_rate (86, 2) 2019-01-01 2026-02-01


...

,date,mortgage_ddu_count_monthly
84,2026-01-01,41.86
85,2026-02-01,18.17


mortgage_ddu_volume (86, 2) 2019-01-01 2026-02-01


,date,mortgage_ddu_volume_mln_rub_monthly
84,2026-01-01,373.89
85,2026-02-01,155.07


mortgage_ddu_rate (86, 2) 2019-01-01 2026-02-01


,date,mortgage_ddu_rate_pct_monthly
84,2026-01-01,7.24
85,2026-02-01,8.89


In [7]:

# Агрегация monthly -> quarter
mortgage_q_frames = []

mortgage_q_frames.append(
    aggregate_monthly_to_quarter(
        mortgage_monthly["mortgage_total_count"],
        "mortgage_total_count_monthly",
        "mortgage_total_count_q",
        how="sum",
    )
)

mortgage_q_frames.append(
    aggregate_monthly_to_quarter(
        mortgage_monthly["mortgage_total_volume"],
        "mortgage_total_volume_mln_rub_monthly",
        "mortgage_total_volume_mln_rub_q",
        how="sum",
    )
)

mortgage_q_frames.append(
    aggregate_monthly_to_quarter(
        mortgage_monthly["mortgage_total_rate"],
        "mortgage_total_rate_pct_monthly",
        "mortgage_total_rate_pct_q",
        how="mean",
    )
)

mortgage_q_frames.append(
    aggregate_monthly_to_quarter(
        mortgage_monthly["mortgage_ddu_count"],
        "mortgage_ddu_count_monthly",
        "mortgage_ddu_count_q",
        how="sum",
    )
)

mortgage_q_frames.append(
    aggregate_monthly_to_quarter(
        mortgage_monthly["mortgage_ddu_volume"],
        "mortgage_ddu_volume_mln_rub_monthly",
        "mortgage_ddu_volume_mln_rub_q",
        how="sum",
    )
)

mortgage_q_frames.append(
    aggregate_monthly_to_quarter(
        mortgage_monthly["mortgage_ddu_rate"],
        "mortgage_ddu_rate_pct_monthly",
        "mortgage_ddu_rate_pct_q",
        how="mean",
    )
)

mortgage_q = outer_merge_on_quarter(mortgage_q_frames)

# Дополнительные доли/отношения
if {"mortgage_ddu_count_q", "mortgage_total_count_q"}.issubset(mortgage_q.columns):
    mortgage_q["mortgage_ddu_count_share_q"] = mortgage_q["mortgage_ddu_count_q"] / mortgage_q["mortgage_total_count_q"].replace(0, np.nan)
if {"mortgage_ddu_volume_mln_rub_q", "mortgage_total_volume_mln_rub_q"}.issubset(mortgage_q.columns):
    mortgage_q["mortgage_ddu_volume_share_q"] = mortgage_q["mortgage_ddu_volume_mln_rub_q"] / mortgage_q["mortgage_total_volume_mln_rub_q"].replace(0, np.nan)

display(mortgage_q.tail())


,quarter,mortgage_total_count_q,mortgage_total_count_q_qoq_change,mortgage_total_count_q_log_return_q,mortgage_total_volume_mln_rub_q,mortgage_total_volume_mln_rub_q_qoq_change,mortgage_total_volume_mln_rub_q_log_return_q,mortgage_total_rate_pct_q,mortgage_total_rate_pct_q_qoq_change,mortgage_total_rate_pct_q_log_return_q,mortgage_ddu_count_q,mortgage_ddu_count_q_qoq_change,mortgage_ddu_count_q_log_return_q,mortgage_ddu_volume_mln_rub_q,mortgage_ddu_volume_mln_rub_q_qoq_change,mortgage_ddu_volume_mln_rub_q_log_return_q,mortgage_ddu_rate_pct_q,mortgage_ddu_rate_pct_q_qoq_change,mortgage_ddu_rate_pct_q_log_return_q,mortgage_ddu_count_share_q,mortgage_ddu_volume_share_q
49,2025Q2,145.86,15.83,0.114882,1109.24,184.33,0.181734,8.550000,0.290000,0.034507,87.92,1.01,0.011554,730.70,55.61,0.079157,6.926667,0.366667,0.054388,0.602770,0.658739
50,2025Q3,175.60,29.74,0.185561,1340.32,231.08,0.189233,8.910000,0.360000,0.041243,103.05,15.13,0.158787,873.22,142.52,0.178185,7.356667,0.430000,0.060228,0.586845,0.651501
51,2025Q4,268.22,92.62,0.423599,2123.89,783.57,0.460341,9.026667,0.116667,0.013009,162.30,59.25,0.454232,1410.78,537.56,0.479710,7.293333,-0.063333,-0.008646,0.605100,0.664243
52,2026Q1,115.48,-152.74,-0.842710,907.25,-1216.64,-0.850587,10.155000,1.128333,0.117783,60.03,-102.27,-0.994602,528.96,-881.82,-0.980985,8.065000,0.771667,0.100573,0.519830,0.583037
53,2026Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. MOEX ISS: индекс московской недвижимости Домклик (MREDC) и другие индексы

In [8]:

def fetch_moex_index_history(secid: str, start: str = START_DATE, end: Optional[str] = END_DATE) -> pd.DataFrame:
    end_eff = pd.Timestamp.today().strftime("%Y-%m-%d") if end is None else pd.Timestamp(end).strftime("%Y-%m-%d")
    start_eff = pd.Timestamp(start).strftime("%Y-%m-%d")

    rows = []
    start_offset = 0
    page_size = 100

    while True:
        url = MOEX_ISS_HISTORY_URL.format(secid=secid)
        params = {
            "from": start_eff,
            "till": end_eff,
            "start": start_offset,
        }
        r = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        payload = r.json()

        hist = payload.get("history", {})
        cols = hist.get("columns", [])
        data = hist.get("data", [])
        if not data:
            break
        batch = pd.DataFrame(data, columns=cols)
        rows.append(batch)

        cursor = payload.get("history.cursor", {})
        cursor_data = cursor.get("data", [])
        if not cursor_data:
            break
        total = cursor_data[0][1]
        page_size = cursor_data[0][2]
        start_offset += page_size
        if start_offset >= total:
            break

    if not rows:
        return pd.DataFrame(columns=["date", f"{secid.lower()}_close"])

    df = pd.concat(rows, ignore_index=True)
    df["date"] = pd.to_datetime(df["TRADEDATE"], errors="coerce")
    out = df[["date", "CLOSE"]].rename(columns={"CLOSE": f"{secid.lower()}_close"}).copy()
    out = out.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return clip_date_range(out)


moex_index_q_frames = []
for secid in MOEX_INDEX_SECIDS:
    daily = fetch_moex_index_history(secid)
    q = aggregate_daily_to_quarter(daily, f"{secid.lower()}_close", f"{secid.lower()}")
    moex_index_q_frames.append(q)
    print(secid, daily.shape)
    display(q.tail(2))


MREDC (484, 2)


,quarter,mredc_avg_q,mredc_eoq,mredc_min_q,mredc_max_q,mredc_obs_n,mredc_log_return_q,mredc_qoq_change
37,2026Q1,338172.752308,345749.81,331227.72,345749.81,13,0.046473,15700.38
38,2026Q2,348740.130000,348740.13,348740.13,348740.13,1,0.008612,2990.32


IMOEX (3327, 2)


,quarter,imoex_avg_q,imoex_eoq,imoex_min_q,imoex_max_q,imoex_obs_n,imoex_log_return_q,imoex_qoq_change
52,2026Q1,2788.210000,2776.37,2696.92,2888.68,60,0.003518,9.75
53,2026Q2,2770.116667,2760.70,2760.70,2775.24,3,-0.005660,-15.67


RGBI (3329, 2)


,quarter,rgbi_avg_q,rgbi_eoq,rgbi_min_q,rgbi_max_q,rgbi_obs_n,rgbi_log_return_q,rgbi_qoq_change
52,2026Q1,117.854000,119.16,115.56,119.89,60,0.009105,1.08
53,2026Q2,119.113333,119.02,119.02,119.17,3,-0.001176,-0.14


## 5. Мосстат: поиск актуальных XLSX по странице и парсинг в кварталы

In [9]:

def find_first_matching_xlsx(page_url: str, include_patterns: list[str]) -> str:
    r = SESSION.get(page_url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    candidates = []
    for a in soup.find_all("a", href=True):
        href = a.get("href")
        text = a.get_text(" ", strip=True)
        blob = f"{text} {href}".lower()
        if ".xlsx" not in href.lower() and ".xls" not in href.lower():
            continue
        if all(re.search(pat, blob, flags=re.I) for pat in include_patterns):
            candidates.append(urljoin(page_url, href))

    if not candidates:
        raise RuntimeError(f"Не нашли XLS/XLSX на странице {page_url} по паттернам {include_patterns}")
    return candidates[0]


def _guess_year_quarter_pairs(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Пробуем найти матрицу, где годы и кварталы лежат в таблице. Функция не идеальна,
    но обычно хорошо работает для компактных xlsx Мосстата.
    """
    txt = raw.copy().astype(str)
    txt = txt.replace("nan", np.nan)

    # Ищем годы в верхних строках
    year_positions = []
    for i in range(min(8, raw.shape[0])):
        for j in range(raw.shape[1]):
            cell = str(raw.iat[i, j]).strip()
            if re.fullmatch(r"20\d{2}", cell):
                year_positions.append((i, j, int(cell)))

    # Ищем квартальные подписи
    quarter_rows = []
    for i in range(raw.shape[0]):
        row_blob = " | ".join(map(str, raw.iloc[i].tolist())).lower()
        if "кварт" in row_blob:
            quarter_rows.append(i)

    pairs = []
    if year_positions and quarter_rows:
        for i in quarter_rows:
            row = raw.iloc[i]
            for j, val in enumerate(row):
                qtxt = str(val).lower()
                qm = re.search(r"([ivx]+)\s*кварт", qtxt)
                if qm:
                    roman = qm.group(1).upper()
                    qmap = {"I":1, "II":2, "III":3, "IV":4}
                    qnum = qmap.get(roman)
                    if qnum is None:
                        continue
                    # Ищем ближайший год сверху в этой же колонке
                    col_year = None
                    for yi, yj, yy in year_positions:
                        if yj == j and yi < i:
                            col_year = yy
                    if col_year is not None:
                        pairs.append({"quarter": f"{col_year}Q{qnum}", "row_idx": i, "col_idx": j})
    return pd.DataFrame(pairs)


def parse_mosstat_primary_price_index(url: str) -> pd.DataFrame:
    xls = pd.ExcelFile(url)
    rows = []
    for sheet in xls.sheet_names:
        raw = pd.read_excel(url, sheet_name=sheet, header=None)
        # Ищем строку "Все типы квартир"
        target_rows = raw.index[
            raw.apply(lambda r: r.astype(str).str.contains("Все типы квартир", case=False, na=False).any(), axis=1)
        ].tolist()
        if not target_rows:
            continue

        pairs = _guess_year_quarter_pairs(raw)
        if pairs.empty:
            continue

        # Сценарий 1: кварталы расположены по колонкам, target row содержит сами значения
        target_idx = target_rows[0]
        for _, rec in pairs.iterrows():
            j = int(rec["col_idx"])
            value = raw.iat[target_idx, j]
            rows.append({"quarter": rec["quarter"], "mosstat_primary_price_index_q": pd.to_numeric(value, errors="coerce")})

    out = pd.DataFrame(rows).dropna().drop_duplicates(subset=["quarter"]).sort_values("quarter")
    if out.empty:
        raise RuntimeError("Не удалось распарсить индекс цен первичного жилья из xlsx Мосстата")
    out["mosstat_primary_price_index_q_qoq_change"] = out["mosstat_primary_price_index_q"].diff()
    return out


def parse_mosstat_cpi(url: str) -> pd.DataFrame:
    raw = pd.read_excel(url, header=None)
    years = [int(x) for x in pd.Series(raw.iloc[3].tolist()).astype(str).str.extract(r"(20\d{2})", expand=False).dropna().unique()]
    rows = []
    # Обычно месяцы идут строками, годы — колонками
    for i in range(raw.shape[0]):
        month_ts = parse_ru_month_year(raw.iat[i, 0])
        if pd.notna(month_ts):
            rows.append({"date": month_ts, "cpi_moscow_monthly": pd.to_numeric(raw.iat[i, 1], errors="coerce")})

    # fallback: вытягиваем из табличного блока месяц x год
    if not rows:
        month_rows = []
        for i in range(raw.shape[0]):
            month_name = str(raw.iat[i, 0]).strip().lower().replace("ё", "е")
            if month_name in RU_MONTHS:
                month_rows.append(i)
        year_cols = []
        for j in range(raw.shape[1]):
            cell = str(raw.iat[3, j]).strip()
            if re.fullmatch(r"20\d{2}", cell):
                year_cols.append((j, int(cell)))
        for i in month_rows:
            month = RU_MONTHS[str(raw.iat[i, 0]).strip().lower().replace("ё", "е")]
            for j, year in year_cols:
                value = pd.to_numeric(str(raw.iat[i, j]).replace(",", "."), errors="coerce")
                rows.append({"date": pd.Timestamp(year=year, month=month, day=1), "cpi_moscow_monthly": value})

    out = pd.DataFrame(rows).dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"])
    out = clip_date_range(out)
    q = aggregate_monthly_to_quarter(out, "cpi_moscow_monthly", "cpi_moscow_q", how="mean")
    return q


def parse_mosstat_housing_completion(url: str) -> pd.DataFrame:
    raw = pd.read_excel(url, header=None)
    rows = []
    # Ищем строки вида 'январь-март 2024' / 'I квартал 2024' / похожие
    for i in range(raw.shape[0]):
        blob = " ".join(map(str, raw.iloc[i].tolist())).lower().replace("ё", "е")
        qm = re.search(r"([ivx]+)\s*квартал\s*(20\d{2})", blob)
        if qm:
            roman = qm.group(1).upper()
            qmap = {"I":1, "II":2, "III":3, "IV":4}
            qnum = qmap.get(roman)
            year = int(qm.group(2))
            # Берём первое числовое значение в строке как тыс. кв. м
            vals = pd.to_numeric(pd.Series(raw.iloc[i].tolist()), errors="coerce").dropna()
            if not vals.empty:
                rows.append({"quarter": f"{year}Q{qnum}", "housing_completion_ths_sqm_q": vals.iloc[0]})

    # fallback: если в xlsx нет явных кварталов, но есть накопительные периоды январь-март / январь-июнь / ...
    if not rows:
        period_map = {
            "январь-март": 1,
            "январь-июнь": 2,
            "январь-сентябрь": 3,
            "январь-декабрь": 4,
        }
        for i in range(raw.shape[0]):
            blob = " ".join(map(str, raw.iloc[i].tolist())).lower().replace("ё", "е")
            for pat, qnum in period_map.items():
                m = re.search(pat + r"\s*(20\d{2})", blob)
                if m:
                    year = int(m.group(1))
                    vals = pd.to_numeric(pd.Series(raw.iloc[i].tolist()), errors="coerce").dropna()
                    if not vals.empty:
                        rows.append({"quarter": f"{year}Q{qnum}", "housing_completion_ths_sqm_cum": vals.iloc[0]})

        tmp = pd.DataFrame(rows).drop_duplicates(subset=["quarter"]).sort_values("quarter")
        if not tmp.empty and "housing_completion_ths_sqm_cum" in tmp.columns:
            out = tmp.copy()
            out["year"] = out["quarter"].str[:4]
            out["housing_completion_ths_sqm_q"] = out.groupby("year")["housing_completion_ths_sqm_cum"].diff()
            mask_q1 = out["quarter"].str.endswith("Q1")
            out.loc[mask_q1, "housing_completion_ths_sqm_q"] = out.loc[mask_q1, "housing_completion_ths_sqm_cum"]
            return out[["quarter", "housing_completion_ths_sqm_q"]].dropna(subset=["housing_completion_ths_sqm_q"])

    out = pd.DataFrame(rows).drop_duplicates(subset=["quarter"]).sort_values("quarter")
    if out.empty:
        raise RuntimeError("Не удалось распарсить ввод жилья из xlsx Мосстата")
    return out


In [10]:

# Пытаемся найти и распарсить Мосстат.
# Если сайт/структура временно поменялись, ноутбук не падает целиком — просто пишет предупреждение.
mosstat_frames = {}

try:
    primary_xlsx = find_first_matching_xlsx(MOSSTAT_PAGES["primary_price_index"], MOSSTAT_PATTERNS["primary_price_index"])
    print("PRIMARY XLSX:", primary_xlsx)
    mosstat_frames["primary_price_index"] = parse_mosstat_primary_price_index(primary_xlsx)
    display(mosstat_frames["primary_price_index"].tail())
except Exception as e:
    print("[WARN] primary_price_index parser failed:", repr(e))

try:
    cpi_xlsx = find_first_matching_xlsx(MOSSTAT_PAGES["cpi"], MOSSTAT_PATTERNS["cpi"])
    print("CPI XLSX:", cpi_xlsx)
    mosstat_frames["cpi"] = parse_mosstat_cpi(cpi_xlsx)
    display(mosstat_frames["cpi"].tail())
except Exception as e:
    print("[WARN] cpi parser failed:", repr(e))

try:
    completion_xlsx = find_first_matching_xlsx(MOSSTAT_PAGES["housing_completion"], MOSSTAT_PATTERNS["housing_completion"])
    print("HOUSING COMPLETION XLSX:", completion_xlsx)
    mosstat_frames["housing_completion"] = parse_mosstat_housing_completion(completion_xlsx)
    display(mosstat_frames["housing_completion"].tail())
except Exception as e:
    print("[WARN] housing_completion parser failed:", repr(e))


[WARN] primary_price_index parser failed: SSLError(MaxRetryError("HTTPSConnectionPool(host='77.rosstat.gov.ru', port=443): Max retries exceeded with url: /folder/64508 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))"))
[WARN] cpi parser failed: SSLError(MaxRetryError("HTTPSConnectionPool(host='77.rosstat.gov.ru', port=443): Max retries exceeded with url: /folder/64640 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))"))
[WARN] housing_completion parser failed: SSLError(MaxRetryError("HTTPSConnectionPool(host='77.rosstat.gov.ru', port=443): Max retries exceeded with url: /folder/64519 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))"))


## 6. Опциональные policy-флаги по льготной ипотеке

In [11]:

# Здесь intentionally ручной календарь: события редкие, а сами флаги проще поддерживать вручную,
# чем парсить новостные страницы при каждом запуске.
# Основной количественный сигнал спроса всё равно лучше брать через фактические ипотечные объёмы/ставки.

policy = make_quarter_spine()
policy["quarter_start"] = pd.PeriodIndex(policy["quarter"], freq="Q").start_time
policy["quarter_end"]   = pd.PeriodIndex(policy["quarter"], freq="Q").end_time

# Массовая льготная ипотека ('Господдержка'): с 17.04.2020 по 01.07.2024
policy["subsidized_mortgage_2020_active_q"] = (
    (policy["quarter_end"] >= pd.Timestamp("2020-04-17")) &
    (policy["quarter_start"] < pd.Timestamp("2024-07-01"))
).astype(int)

# Новые правила семейной ипотеки / продление до 2030: используем как structural dummy с 2024Q3
policy["family_mortgage_new_rules_active_q"] = (policy["quarter_start"] >= pd.Timestamp("2024-07-01")).astype(int)

# Новые правила IT-ипотеки / продление до 2030: тоже structural dummy с 2024Q3
policy["it_mortgage_new_rules_active_q"] = (policy["quarter_start"] >= pd.Timestamp("2024-07-01")).astype(int)

policy = policy.drop(columns=["quarter_start", "quarter_end"])
display(policy.tail())


,quarter,subsidized_mortgage_2020_active_q,family_mortgage_new_rules_active_q,it_mortgage_new_rules_active_q
49,2025Q2,0,1,1
50,2025Q3,0,1,1
51,2025Q4,0,1,1
52,2026Q1,0,1,1
53,2026Q2,0,1,1


## 7. Склейка всех источников в единый quarterly CSV

In [12]:

all_frames = [
    key_rate_q,
    usd_q,
    eur_q,
    mortgage_q,
    *moex_index_q_frames,
    policy,
]

for v in mosstat_frames.values():
    all_frames.append(v)

macro_q = outer_merge_on_quarter(all_frames).sort_values("quarter").reset_index(drop=True)

# Полезные derived-features
if {"key_rate_avg_q", "cpi_moscow_q"}.issubset(macro_q.columns):
    macro_q["real_policy_rate_q"] = macro_q["key_rate_avg_q"] - macro_q["cpi_moscow_q"]

if {"mortgage_total_volume_mln_rub_q", "mortgage_total_count_q"}.issubset(macro_q.columns):
    macro_q["mortgage_avg_ticket_mln_rub_q"] = macro_q["mortgage_total_volume_mln_rub_q"] / macro_q["mortgage_total_count_q"].replace(0, np.nan)

if {"mortgage_ddu_volume_mln_rub_q", "mortgage_ddu_count_q"}.issubset(macro_q.columns):
    macro_q["mortgage_ddu_avg_ticket_mln_rub_q"] = macro_q["mortgage_ddu_volume_mln_rub_q"] / macro_q["mortgage_ddu_count_q"].replace(0, np.nan)

# Сохраняем
macro_q.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_CSV)
print("Shape:", macro_q.shape)
display(macro_q.tail(12))


coverage_df = build_coverage_table(macro_q)
coverage_df.to_csv(OUTPUT_COVERAGE_CSV, index=False, encoding="utf-8-sig")
print("Saved coverage:", OUTPUT_COVERAGE_CSV)
display(coverage_df)


Saved: cashflow_project\data\macro\macro_quarterly_Moscow.csv
Shape: (54, 68)


,quarter,key_rate_avg_q,key_rate_eoq,key_rate_min_q,key_rate_max_q,key_rate_obs_n,key_rate_log_return_q,key_rate_qoq_change,usd_rub_avg_q,usd_rub_eoq,usd_rub_min_q,usd_rub_max_q,usd_rub_obs_n,usd_rub_log_return_q,usd_rub_qoq_change,eur_rub_avg_q,eur_rub_eoq,eur_rub_min_q,eur_rub_max_q,eur_rub_obs_n,eur_rub_log_return_q,eur_rub_qoq_change,mortgage_total_count_q,mortgage_total_count_q_qoq_change,mortgage_total_count_q_log_return_q,mortgage_total_volume_mln_rub_q,mortgage_total_volume_mln_rub_q_qoq_change,mortgage_total_volume_mln_rub_q_log_return_q,mortgage_total_rate_pct_q,mortgage_total_rate_pct_q_qoq_change,mortgage_total_rate_pct_q_log_return_q,mortgage_ddu_count_q,mortgage_ddu_count_q_qoq_change,mortgage_ddu_count_q_log_return_q,mortgage_ddu_volume_mln_rub_q,mortgage_ddu_volume_mln_rub_q_qoq_change,mortgage_ddu_volume_mln_rub_q_log_return_q,mortgage_ddu_rate_pct_q,mortgage_ddu_rate_pct_q_qoq_change,mortgage_ddu_rate_pct_q_log_return_q,mortgage_ddu_count_share_q,mortgage_ddu_volume_share_q,mredc_avg_q,mredc_eoq,mredc_min_q,mredc_max_q,mredc_obs_n,mredc_log_return_q,mredc_qoq_change,imoex_avg_q,imoex_eoq,imoex_min_q,imoex_max_q,imoex_obs_n,imoex_log_return_q,imoex_qoq_change,rgbi_avg_q,rgbi_eoq,rgbi_min_q,rgbi_max_q,rgbi_obs_n,rgbi_log_return_q,rgbi_qoq_change,subsidized_mortgage_2020_active_q,family_mortgage_new_rules_active_q,it_mortgage_new_rules_active_q,mortgage_avg_ticket_mln_rub_q,mortgage_ddu_avg_ticket_mln_rub_q
42,2023Q3,10.179348,13.0,7.5,13.0,92.0,0.550046,5.5,94.200117,97.4147,88.3844,101.0399,66,0.112677,10.3806,102.544355,103.1631,96.0195,110.6847,66,0.081328,8.0579,400.75,108.94,0.317235,2925.69,764.03,0.302654,8.126667,-0.410000,-0.049220,190.99,77.18,0.517691,1492.45,585.09,0.497635,6.233333,-0.220000,-0.034686,0.476581,0.510119,270072.423846,277555.51,264312.51,277555.51,13.0,0.042700,11602.16,3060.911231,3133.26,2793.93,3268.97,65,0.113394,335.89,124.575538,119.34,119.34,128.35,65,-0.073252,-9.07,1,0,0,7.300536,7.814283
43,2023Q4,14.521739,16.0,13.0,16.0,92.0,0.207639,3.0,92.753353,89.6883,87.8701,101.3598,64,-0.082637,-7.7264,99.762511,99.1919,96.1475,107.0322,64,-0.039255,-3.9712,376.92,-23.83,-0.061305,2689.66,-236.03,-0.084116,8.256667,0.130000,0.015870,198.15,7.16,0.036803,1578.48,86.03,0.056043,6.343333,0.110000,0.017493,0.525708,0.586870,283709.250769,285189.63,278455.53,286592.02,13.0,0.027133,7634.12,3167.731692,3099.11,3008.84,3269.27,65,-0.010959,-34.15,120.076462,121.05,117.32,122.46,65,0.014227,1.71,1,0,0,7.135891,7.966086
44,2024Q1,16.000000,16.0,16.0,16.0,91.0,0.000000,0.0,90.800246,92.3660,87.6457,92.7761,57,0.029419,2.6777,98.498472,99.5299,95.6007,100.6139,57,0.003402,0.3380,193.64,-183.28,-0.666032,1335.95,-1353.71,-0.699772,9.086667,0.830000,0.095787,90.48,-107.67,-0.783895,744.06,-834.42,-0.752096,6.650000,0.306667,0.047212,0.467259,0.556952,290227.038462,294482.13,284198.72,296472.59,13.0,0.032064,9292.50,3231.124918,3332.53,3130.23,3332.60,61,0.072617,233.42,119.178525,115.38,115.07,121.49,61,-0.047973,-5.67,1,0,0,6.899143,8.223475
45,2024Q2,16.000000,16.0,16.0,16.0,91.0,0.000000,0.0,90.480057,85.7480,82.6282,94.3242,60,-0.074346,-6.6180,97.483408,92.4184,89.0914,101.2333,60,-0.074132,-7.1115,311.16,117.52,0.474306,2148.96,813.01,0.475341,8.443333,-0.643333,-0.073431,166.41,75.93,0.609326,1332.47,588.41,0.582668,6.556667,-0.093333,-0.014135,0.534805,0.620053,298621.627692,303081.32,294218.03,303081.32,13.0,0.028783,8599.19,3343.039524,3154.36,3027.47,3501.89,63,-0.054946,-178.17,111.110794,106.80,104.97,115.23,63,-0.077273,-8.58,1,0,0,6.906286,8.007151
46,2024Q3,17.554348,19.0,16.0,19.0,92.0,0.171850,3.0,89.233694,92.7126,84.9471,92.9200,65,0.078092,6.9646,98.015091,103.4694,92.8291,103.4758,65,0.112950,11.0510,179.99,-131.17,-0.547406,1257.58,-891.38,-0.535795,10.376667,1.933333,0.206183,73.13,-93.28,-0.822216,648.44,-684.03,-0.720220,7.183333,0.626667,0.091281,0.406300,0.515625,297622.034615,299747.70,291119.90,304036.89,13.0,-0.011060,-3333.62,2855.608182,2857.56,2524.38,3217.29,6

Saved coverage: cashflow_project\data\macro\macro_quarterly_Moscow_coverage.csv


,feature,first_quarter,last_quarter,non_null_quarters
0,eur_rub_avg_q,2013Q1,2026Q2,54
1,eur_rub_eoq,2013Q1,2026Q2,54
2,eur_rub_max_q,2013Q1,2026Q2,54
3,eur_rub_min_q,2013Q1,2026Q2,54
4,eur_rub_obs_n,2013Q1,2026Q2,54
...,...,...,...,...
62,mortgage_total_count_q_qoq_change,2019Q2,2026Q1,28
63,mortgage_total_rate_pct_q_log_return_q,2019Q2,2026Q1,28
64,mortgage_total_rate_pct_q_qoq_change,2019Q2,2026Q1,28
65,mortgage_total_volume_mln_rub_q_log_return_q,2019Q2,2026Q1,28



## 8. Что проверить перед использованием в market block

Перед тем как использовать этот CSV в модели, я бы сделал ещё 3 sanity-check'а:

1. **Сверить уровни и даты**
   - `MREDC` должен быть недельным и агрегироваться в квартал без дыр.
   - ипотека ЦБ должна давать осмысленные квартальные суммы/средние.

2. **Проверить мультиколлинеарность**
   - `key_rate`, `mortgage_rate`, `usd_rub`, `RGBI` часто частично дублируют один и тот же стресс-фактор.
   - для MVP лучше держать не слишком широкий список признаков.

3. **Не отдавать все сырые макро-фичи прямо в deal-level hedonic**
   - лучше сначала обучить отдельный квартальный market model,
   - а в final hedonic передавать уже `market_state` / `market_log_price` / `market_return`.


In [13]:
print("Head of macro dataset:")
display(macro_q.head(16))

print("Rows with key_rate present:")
display(macro_q.loc[macro_q["key_rate_avg_q"].notna(), ["quarter", "key_rate_avg_q", "key_rate_eoq"]].head(12))

print("Coverage summary for mortgage-related series:")
display(coverage_df.loc[coverage_df["feature"].str.contains("mortgage", na=False)].reset_index(drop=True))

Head of macro dataset:


,quarter,key_rate_avg_q,key_rate_eoq,key_rate_min_q,key_rate_max_q,key_rate_obs_n,key_rate_log_return_q,key_rate_qoq_change,usd_rub_avg_q,usd_rub_eoq,usd_rub_min_q,usd_rub_max_q,usd_rub_obs_n,usd_rub_log_return_q,usd_rub_qoq_change,eur_rub_avg_q,eur_rub_eoq,eur_rub_min_q,eur_rub_max_q,eur_rub_obs_n,eur_rub_log_return_q,eur_rub_qoq_change,mortgage_total_count_q,mortgage_total_count_q_qoq_change,mortgage_total_count_q_log_return_q,mortgage_total_volume_mln_rub_q,mortgage_total_volume_mln_rub_q_qoq_change,mortgage_total_volume_mln_rub_q_log_return_q,mortgage_total_rate_pct_q,mortgage_total_rate_pct_q_qoq_change,mortgage_total_rate_pct_q_log_return_q,mortgage_ddu_count_q,mortgage_ddu_count_q_qoq_change,mortgage_ddu_count_q_log_return_q,mortgage_ddu_volume_mln_rub_q,mortgage_ddu_volume_mln_rub_q_qoq_change,mortgage_ddu_volume_mln_rub_q_log_return_q,mortgage_ddu_rate_pct_q,mortgage_ddu_rate_pct_q_qoq_change,mortgage_ddu_rate_pct_q_log_return_q,mortgage_ddu_count_share_q,mortgage_ddu_volume_share_q,mredc_avg_q,mredc_eoq,mredc_min_q,mredc_max_q,mredc_obs_n,mredc_log_return_q,mredc_qoq_change,imoex_avg_q,imoex_eoq,imoex_min_q,imoex_max_q,imoex_obs_n,imoex_log_return_q,imoex_qoq_change,rgbi_avg_q,rgbi_eoq,rgbi_min_q,rgbi_max_q,rgbi_obs_n,rgbi_log_return_q,rgbi_qoq_change,subsidized_mortgage_2020_active_q,family_mortgage_new_rules_active_q,it_mortgage_new_rules_active_q,mortgage_avg_ticket_mln_rub_q,mortgage_ddu_avg_ticket_mln_rub_q
0,2013Q1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.415821,31.0834,29.9251,31.0834,57,NaN,NaN,40.185089,39.8023,39.6385,40.8674,57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1502.718448,1438.57,1417.00,1562.93,58,NaN,NaN,138.628276,137.67,137.02,139.82,58,NaN,NaN,0,0,0,NaN,NaN
1,2013Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.660954,32.7090,30.8814,32.9097,59,0.050976,1.6256,41.322441,42.7180,39.8168,43.3526,59,0.070696,2.9157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1368.450161,1330.46,1281.89,1448.42,62,-0.078125,-108.11,137.769839,133.47,129.90,141.65,62,-0.030983,-4.20,0,0,0,NaN,NaN
2,2013Q3,5.500000,5.5,5.5,5.5,14.0,NaN,NaN,32.798492,32.3451,31.5892,33.4656,65,-0.011188,-0.3639,43.428386,43.6497,42.1033,44.3879,65,0.021576,0.9317,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1403.665758,1462.82,1333.35,1487.19,66,0.094841,132.36,134.677879,134.86,132.50,136.41,66,0.010360,1.39,0,0,0,NaN,NaN
3,2013Q4,5.500000,5.5,5.5,5.5,92.0,0.000000,0.0,32.543862,32.7292,31.6618,33.2632,65,0.011805,0.3841,44.292309,44.9699,43.5123,45.3688,65,0.029797,1.3202,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1492.352187,1504.08,1429.25,1533.61,64,0.027815,41.26,134.155781,133.27,132.02,135.96,64,-0.011860,-1.59,0,0,0,NaN,NaN
4,2014Q1,5.983333,7.0,5.5,7.0,90.0,0.241162,1.5,35.143616,35.6871,32.6587,36.6505,57,0.086522,2.9579,48.171132,49.0519,45.0559,50.9442,57,0.086886,4.0820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1422.402542,1369.29,1237.43,1507.49,59,-0.093889,-134.79,129.569492,126.83,123.40,133.28,59,-0.049530,-6.44,0,0,0,NaN,NaN
5,2014Q2,7.351648,7.5,7.0,7.5,91.0,0.068993,0.5,35.024245,33.6306,33.6306,36.0813,60,-0.059353,-2.0565,48.059638,45.8251,45.8251,49.8860,60,-0.068047,-3.2268,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1402.642131,1476.38,1280.12,1518.83,61,0.075301,107.09,126.751639,128.50,123.63,128.78,61,0.013081,1.67,0,0,0,NaN,NaN
6,2014Q3,7.853261,8.0,7.5,8.0,92.0,0.064539,0.5,36.162382,39.3866,33.8353,39.3866,66,0.157989,5.7560,47.980777,49.9540,46.1649,50.0582,66,0.086271,4.1289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1432.290152,1411.07,1333.53,1516.75,66,-0.045245,-65.3

Rows with key_rate present:


,quarter,key_rate_avg_q,key_rate_eoq
2,2013Q3,5.500000,5.5
3,2013Q4,5.500000,5.5
4,2014Q1,5.983333,7.0
5,2014Q2,7.351648,7.5
6,2014Q3,7.853261,8.0
7,2014Q4,10.277174,17.0
8,2015Q1,15.533333,14.0
9,2015Q2,12.895604,11.5
10,2015Q3,11.179348,11.0
11,2015Q4,11.000000,11.0


Coverage summary for mortgage-related series:


,feature,first_quarter,last_quarter,non_null_quarters
0,family_mortgage_new_rules_active_q,2013Q1,2026Q2,54
1,it_mortgage_new_rules_active_q,2013Q1,2026Q2,54
2,subsidized_mortgage_2020_active_q,2013Q1,2026Q2,54
3,mortgage_avg_ticket_mln_rub_q,2019Q1,2026Q1,29
4,mortgage_ddu_avg_ticket_mln_rub_q,2019Q1,2026Q1,29
5,mortgage_ddu_count_q,2019Q1,2026Q1,29
6,mortgage_ddu_count_share_q,2019Q1,2026Q1,29
7,mortgage_ddu_rate_pct_q,2019Q1,2026Q1,29
8,mortgage_ddu_volume_mln_rub_q,2019Q1,2026Q1,29
9,mortgage_ddu_volume_share_q,2019Q1,2026Q1,29
